<a href="https://colab.research.google.com/github/Nihat0311/NLP/blob/main/Grammar_correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = load_dataset("agentlans/grammar-correction")

print("\nDataset:")
print(dataset)

print("\nDataset nümunə:")
print(dataset['train'][0])

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

train.jsonl.zst:   0%|          | 0.00/6.91M [00:00<?, ?B/s]

validation.jsonl.zst:   0%|          | 0.00/1.77M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/25000 [00:00<?, ? examples/s]


Dataset:
DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 100000
    })
    validation: Dataset({
        features: ['input', 'output'],
        num_rows: 25000
    })
})

Dataset nümunə:
{'input': 'The Kānvar is a travel, made by devotees of Shiva, known as Kānvarias of Hindu pilgrimage places of Haridwar as well as Gaumukh and Gangotri in to fetch in holy waters of Ganges River.', 'output': 'The Kānvar is a travel, undertaken by devotees of Shiva, which are known as Kānvarias to Hindu pilgrimage places of Haridwar as well as Gaumukh and Gangotri in to fetch holy waters of Ganges River.'}


In [ ]:
model_checkpoint = "t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

prefix = "correct grammar: "

def preprocess_function(examples):
    inputs = [prefix + text for text in examples["input"]]
    targets = examples["output"]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

model.to(device)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
import torch
import gc
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
gc.collect()
torch.cuda.empty_cache()

small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(20000))
small_eval_dataset = tokenized_datasets["validation"].shuffle(seed=42).select(range(2000))

training_args = Seq2SeqTrainingArguments(
    output_dir="./grammar_correction_model",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=True,
    logging_steps=50,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    processing_class=tokenizer,             ğ
    data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.728757,0.160531
2,0.681871,0.158807


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.691675048828125, metrics={'train_runtime': 1232.8109, 'train_samples_per_second': 32.446, 'train_steps_per_second': 1.014, 'total_flos': 6089578905600000.0, 'train_loss': 0.691675048828125, 'epoch': 2.0})

In [ ]:
!pip install evaluate rouge_score absl-py

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=7e50ffeee2fc3114ee3d070c96197e8dd247b51bbb623a721ae0ec9f4a6fd276
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
import evaluate
import numpy as np

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = preds.astype(np.int64)
    valid_token_ids = (preds >= 0) & (preds < tokenizer.vocab_size)
    preds[~valid_token_ids] = tokenizer.pad_token_id

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]
    rouge_result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    bleu_result = bleu_metric.compute(predictions=decoded_preds, references=[[label] for label in decoded_labels])

    return {
        "bleu": bleu_result["bleu"],
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"]
    }

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Bleu,Rouge1,Rouge2,Rougel
1,0.578170,0.160205,0.272926,0.696126,0.591399,0.687536
2,0.627840,0.157106,0.272603,0.696051,0.591129,0.687406


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.5530955627441406, metrics={'train_runtime': 1575.6297, 'train_samples_per_second': 25.387, 'train_steps_per_second': 0.793, 'total_flos': 6089578905600000.0, 'train_loss': 0.5530955627441406, 'epoch': 2.0})

In [ ]:
import torch

def sentence_correction(sentence, model, tokenizer, device):
    model.eval()
    prefix = "correct grammar: "
    input_text = prefix + sentence
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=128,
            num_beams=4,
            early_stopping=True
        )
    predicted_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return predicted_sentence

wrong_sentence = "He go to school yesterday and he buy a apple."

result = sentence_correction(wrong_sentence, model, tokenizer, device)

print(f"Orijinal (Hatalı) Cümle : {wrong_sentence}")
print(f"Modelin Düzeltmesi       : {result}")

Orijinal (Hatalı) Cümle : He go to school yesterday and he buy a apple.
Modelin Düzeltmesi       : He went to school yesterday and he bought an apple.
